# BinaryTask: product, channel и both

Notebook обучает реальные CatBoost-модели на полном синтетическом наборе. В каждом scope используется Optuna с одним trial. `product` строит одну общую модель, `channel` — модель на каждое значение `group`, `both` — оба набора моделей.

In [1]:
from pathlib import Path
from time import perf_counter

import polars as pl
from IPython.display import display

from avatar.automl import BinaryTask, BinaryTaskConfig

WORKSPACE = Path.cwd().parent.resolve()
TRAIN_PATH = WORKSPACE / 'data/binary/train'
VALID_PATH = WORKSPACE / 'data/binary/valid'
TEST_PATH = WORKSPACE / 'data/binary/test'
CAT_COLS = [f'cat_feature_{index}' for index in range(1, 6)]
NUM_COLS = [f'num_feature_{index}' for index in range(1, 6)]
CATBOOST_ONE_TRIAL = {
    'iterations': [16], 'depth': [2], 'l2_leaf_reg': [1.0],
    'bagging_temperature': [0.25], 'learning_rate': [0.1],
}


In [2]:
rows = []
expected_model_counts = {'product': 1, 'channel': 2, 'both': 3}
for scope in ('product', 'channel', 'both'):
    config = BinaryTaskConfig(environment={}, 
        env_type='local', backend='boosting', engine='catboost', device='gpu',
        target_column='target', client_id_column='epk_id', report_month_column='report_month',
        group_column='group', treatment_column='treatment', inverse_treatment=True,
        model_scope=scope, categorical_columns=CAT_COLS, numerical_columns=NUM_COLS,
        hidden_state_columns=['seq_hidden_state'], hyperopt=True, optimization_metric='roc_auc',
        n_trials=1, random_state=42, verbose=False,
        search_space=CATBOOST_ONE_TRIAL,
        output_dir=WORKSPACE / f'outputs/notebook_tests/model_scope/{scope}',
    )
    task = BinaryTask(config)
    started = perf_counter()
    training = task.train(TRAIN_PATH, VALID_PATH)
    train_seconds = perf_counter() - started
    prediction = task.predict(TEST_PATH)
    evaluation = task.evaluate(TEST_PATH, prediction)
    assert len(training.best_params) == expected_model_counts[scope]
    assert prediction.scores.height == 30_000
    rows.append({
        'scope': scope, 'model_count': len(training.best_params),
        'model_keys': ', '.join(training.best_params),
        'validation_roc_auc_mean': sum(training.validation_metrics.values()) / len(training.validation_metrics),
        'test_roc_auc': evaluation.metrics['roc_auc'], 'train_seconds': train_seconds,
    })

calibrated_config = BinaryTaskConfig(environment={}, 
    env_type='local', backend='boosting', engine='catboost', device='gpu',
    target_column='target', client_id_column='epk_id', report_month_column='report_month',
    group_column='group', treatment_column='treatment', inverse_treatment=True,
    model_scope='both', categorical_columns=CAT_COLS, numerical_columns=NUM_COLS,
    hidden_state_columns=['seq_hidden_state'], hyperopt=True, optimization_metric='roc_auc',
    n_trials=1, random_state=42, verbose=False,
    search_space=CATBOOST_ONE_TRIAL,
    calibration_windows={
        '2024-10-01': ['2024-07-01','2024-08-01','2024-09-01'],
        '2024-11-01': ['2024-08-01','2024-09-01'],
        '2024-12-01': ['2024-09-01'],
    },
    output_dir=WORKSPACE / 'outputs/notebook_tests/model_scope/both_calibrated',
)
calibrated_task = BinaryTask(calibrated_config)
calibrated_training = calibrated_task.train(TRAIN_PATH, VALID_PATH)
calibrated_prediction = calibrated_task.predict(TEST_PATH, calibration_paths=[TRAIN_PATH, VALID_PATH])
calibrated_evaluation = calibrated_task.evaluate(TEST_PATH, calibrated_prediction)
assert len(calibrated_training.best_params) == 3
assert calibrated_prediction.raw_scores is not None
assert calibrated_evaluation.calibrated_metrics is not None
assert 'metrics_calibrated' in calibrated_evaluation.excel_paths
rows.append({'scope':'both_calibrated','model_count':3,'model_keys':', '.join(calibrated_training.best_params),'validation_roc_auc_mean':sum(calibrated_training.validation_metrics.values())/3,'test_roc_auc':calibrated_evaluation.calibrated_metrics['roc_auc'],'train_seconds':None})

scope_summary = pl.DataFrame(rows)
display(scope_summary)


2026-08-17 14:39:40,352 INFO run_id=acd6769378a2438696a7437e1eb05b79 Starting action=train task_config=BinaryTaskConfig backend=boosting engine=catboost env_type=local


<workspace>\fmlib-main\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-08-17 14:39:42,673] A new study created in memory with name: no-name-9e9bc69e-8395-4083-adb2-3c4b08d50107


[I 2026-08-17 14:39:45,608] Trial 0 finished with value: 0.7242174460395527 and parameters: {'iterations': 16, 'depth': 2, 'l2_leaf_reg': 1.0, 'bagging_temperature': 0.25, 'learning_rate': 0.1}. Best is trial 0 with value: 0.7242174460395527.


2026-08-17 14:39:45,611 INFO run_id=acd6769378a2438696a7437e1eb05b79 Finished action=train


2026-08-17 14:39:45,617 INFO run_id=0a66fd01e88a4c62ad33b0b1f396b56b Starting action=predict task_config=BinaryTaskConfig backend=boosting engine=catboost env_type=local


...

2026-08-17 14:40:10,742 INFO run_id=555894fe265b46419c7359b67b9fe50a Starting action=predict task_config=BinaryTaskConfig backend=boosting engine=catboost env_type=local


2026-08-17 14:40:11,163 INFO run_id=555894fe265b46419c7359b67b9fe50a Finished action=predict


2026-08-17 14:40:11,169 INFO run_id=2714d92c50914d6f91b2e3337da38e29 Starting action=evaluate task_config=BinaryTaskConfig backend=boosting engine=catboost env_type=local


2026-08-17 14:40:12,775 INFO run_id=2714d92c50914d6f91b2e3337da38e29 Finished action=evaluate


scope,model_count,model_keys,validation_roc_auc_mean,test_roc_auc,train_seconds
str,i64,str,f64,f64,f64
"""product""",1,"""product""",0.724217,0.72347,5.268936
"""channel""",2,"""channel:channel_0, channel:cha…",0.724767,0.724035,4.062546
"""both""",3,"""product, channel:channel_0, ch…",0.72459,0.724035,7.089033
"""both_calibrated""",3,"""product, channel:channel_0, ch…",0.724589,0.724042,null


Итоговая таблица подтверждает число моделей и показывает validation/test ROC AUC для каждого режима. В `both` предсказания маршрутизируются через channel-модели; product-модель остаётся fallback для неизвестного канала.